# SMART Baseline Wrapper (Colab)

Objective:
- Reproduce SMART as an external benchmark baseline via thin wrapper logic.
- Keep orchestration in notebook and metadata/artifact logic in `src/workflows/smart_baseline_flow.py`.
- Persist run metadata and metrics for fair comparison against future Gap-2 variants.

Optional evaluator ingestion:
- Set `WOSAC_OFFICIAL_METRICS_JSON` or `WOSAC_OFFICIAL_METRICS_CSV` before Step 4.


In [ ]:
# Step 1: Sync this wrapper repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-baseline config and validate run context
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-baseline"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
print("Experiment:", EXPERIMENT_SLUG)
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_baseline"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
N_SHARDS = int(RUN.get("n_shards", 1))
SHARD_ID = int(RUN.get("shard_id", 0))
assert PERSIST_ROOT.strip(), "persist_root is required"
assert N_SHARDS >= 1, "n_shards must be >= 1"
assert 0 <= SHARD_ID < N_SHARDS, "shard_id must be in [0, n_shards)"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

print("RUN_TAG:", RUN_TAG)
print("cfg_hash:", cfg_hash)
print("persist_run_dir:", persist_run_dir)


In [ ]:
# Step 3: Sync SMART repo and build executable command plan
from src.workflows import run_smart_baseline_flow

OFFICIAL_METRICS_JSON = os.environ.get("WOSAC_OFFICIAL_METRICS_JSON", "").strip()
OFFICIAL_METRICS_CSV = os.environ.get("WOSAC_OFFICIAL_METRICS_CSV", "").strip()

flow_bundle = run_smart_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path=str(SMART.get("ckpt_path", "")),
    smart_raw_data_root=str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART.get("install_pyg", True)),
    sync_smart_repo=True,
    official_metrics_json=OFFICIAL_METRICS_JSON,
    metrics_csv=OFFICIAL_METRICS_CSV,
)

print("flow_summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("command_plan:")
print(json.dumps(flow_bundle.command_plan, indent=2, sort_keys=True))
print("metrics:")
print(json.dumps(flow_bundle.metrics, indent=2, sort_keys=True))


In [ ]:
# Step 4: Optional command execution (set flags to True when ready)
RUN_SETUP = False
RUN_PREPROCESS = False
RUN_TRAIN = False
RUN_VALIDATE = False

cmds = []
if RUN_SETUP:
    cmds.append(flow_bundle.command_plan["setup_cmd"])
if RUN_PREPROCESS:
    cmds.extend([
        flow_bundle.command_plan["preprocess_train_cmd"],
        flow_bundle.command_plan["preprocess_val_cmd"],
    ])
if RUN_TRAIN:
    cmds.append(flow_bundle.command_plan["train_cmd"])
if RUN_VALIDATE:
    cmds.append(flow_bundle.command_plan["validate_cmd"])

for i, cmd in enumerate(cmds, start=1):
    print(f"[exec {i}/{len(cmds)}] {cmd}")
    subprocess.run(["bash", "-lc", cmd], check=True)

print("Execution block done.")


In [ ]:
# Step 5: Write SMART baseline artifact (repo + Drive)
try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = "unknown"

payload = {
    "run_id": "smart_baseline_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "metrics": dict(flow_bundle.metrics),
    "flow_summary": flow_bundle.summary,
    "command_plan": flow_bundle.command_plan,
    "flow_artifacts": flow_bundle.artifacts,
    "notes": [
        "SMART wrapper baseline artifact for external benchmark comparison.",
        "Set WOSAC_OFFICIAL_METRICS_JSON/CSV to ingest evaluator outputs.",
    ],
    "provenance": {
        "repo_commit": git_commit,
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_metrics_path = Path("experiments/smart-baseline/results/smart_baseline_v0_metrics.json")
repo_metrics_path.parent.mkdir(parents=True, exist_ok=True)
repo_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_metrics_path = persist_run_dir / "outputs" / "smart_baseline_v0_metrics.json"
drive_metrics_path.parent.mkdir(parents=True, exist_ok=True)
drive_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_metrics_path:", repo_metrics_path.resolve())
print("drive_metrics_path:", drive_metrics_path)
payload
